# Oware Model

This notebook is the interactive entry point for the Python Oware model.
The rules currently mirror the C++ OpenSpiel implementation and can be
adjusted in `Model/oware/oware.py` when the Oware rules are
ready.

## 1. Load the game

Importing the module registers the game with OpenSpiel under the name
`oware`.

In [4]:
pip install open-spiel 


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 43.3 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 4.0 MB/s eta 0:00:00


In [7]:
import pyspiel
from Model.oware import oware  # Registers the game

game = pyspiel.load_game("oware")
state = game.new_initial_state()
print(game)
print(state)
print("Legal actions:", state.legal_actions())

ModuleNotFoundError: No module named 'ayo_olopon'

## 2. Board representation

The twelve houses are stored in sowing order:

```text
Player 0:  0  1  2  3  4  5
Player 1:  6  7  8  9 10 11
```

Actions are local to a player's row. For example, action `0` refers to
house `0` for player 0 and house `6` for player 1.

In [ ]:
print("Board:", state.board)
print("Captured:", state.captured)
print("Current player:", state.current_player())
print("Observation size:", len(state.board) + len(state.captured))

## 3. Play one move

The selected action is converted to a physical house, its seeds are
distributed counter-clockwise, and the turn changes.

In [ ]:
action = state.legal_actions()[0]
print("Playing action:", action)
state.apply_action(action)
print(state)

## 4. Inspect an observation for a model

The observer returns normalized house counts followed by normalized
captured scores. This is the vector that can be passed to a neural model.

In [ ]:
observer = game.make_py_observer()
observer.set_from(state, state.current_player())
print(observer.tensor)
print("Observation shape:", observer.tensor.shape)

## 5. Run a random game

This is useful for checking that legal moves, sowing, captures, and
termination work together before connecting a learning algorithm.

In [ ]:
random_state = game.new_initial_state()
while not random_state.is_terminal():
    action = random_state.legal_actions()[0]
    random_state.apply_action(action)

print(random_state)
print("Returns:", random_state.returns())

## Where to adjust the rules

Edit `Model/oware/oware.py`, especially these methods:

- `_legal_actions` — feeding and legal-move rules
- `_distribute_seeds` — sowing direction and source-house behavior
- `_is_grand_slam` — Grand Slam interpretation
- `_capture_from` — capture rules
- `_collect_and_terminate` — end-of-game scoring